## **Load Data**

In [3]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/CustomerSegment/survey.csv")
print(df.shape)

(83, 2)


In [5]:
df = df.dropna()
print(df.shape)

(66, 2)


In [7]:
df.head(4)

,job_title,join_reason
0,Founder,"How to fine tune LLM effectively, and what LLM..."
1,Financial Management & Compliance Advisor,Gain knowledge and skills to be able to develo...
2,Founder in the making,Build an AI product
3,Staff Scientist,Better understanding and skill set in AI to ma...


## **Text Embedding**

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
import numpy  as np
job_embeddings = model.encode(df['job_title'].tolist(),    convert_to_numpy=True)
reason_embeddings = model.encode(df['join_reason'].tolist(),  convert_to_numpy=True)

In [15]:
job_embedding_dim = 12
reason_embedding_dim = 24

In [16]:
job_embeddings = job_embeddings[:, :job_embedding_dim]
reason_embeddings = reason_embeddings[:, :reason_embedding_dim]

In [18]:
embedding_list = [np.concatenate((job_embeddings[i], reason_embeddings[i])) for i in range(len(df))]

In [20]:
col_names = [f"job_embedding-{i+1}" for i in range(job_embeddings.shape[1])] + \
            [f"reason_embedding-{i+1}" for i in range(reason_embeddings.shape[1])]

df_embeddings = pd.DataFrame(embedding_list, columns=col_names)

## **Clustering**

In [22]:
from sklearn.cluster import KMeans

num_segments = 5

clustering = KMeans(
    n_clusters = num_segments,
    random_state=0).fit(df_embeddings)

## **Prompt Building**

In [24]:
prompt_template = lambda markdown_table: f"""You are a business strategist specializing in customer segmentation and profiling. \
Below is a table of survey responses from customers, including their job titles and reasons for joining an AI bootcamp. \
Your task is to analyze the responses and generate a **single predominant customer profile** that represents the most common \
characteristics and motivations across the group.

Please include:
1. Job Title: A single representative job title summarizing those listed in the table
2. Desired Outcomes: A single representative desired outcome synthesizing those listed in the table


### Survey Responses
{markdown_table}        |


### Instructions:
- Keep each section concise (1 sentence or bullet points).
- Use simple language and avoid unnecessary details.
- Total response should be under **100 words**.
- Only include the three items list above (i.e. Job Title, Desired Outcomes, and Key Challenges)


Begin your analysis and generate the customer profile below:
"""

# **Generating with AI**

In [26]:
pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 13.3 MB/s eta 0:00:00


In [28]:
import anthropic

In [29]:
my_sk = ""
client = anthropic.Anthropic(api_key=my_sk)

In [31]:
def generate_avatar(my_sk, df_segment):
    # construct prompt
    prompt = prompt_template(df_segment)

    # make api call
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        temperature=0.5,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # extract response
    customer_avatar = response.content[0].text
    return customer_avatar

In [32]:
for i in range(num_segments):
    survey_data = df[clustering.labels_==i][['job_title', 'join_reason']]
    print("Segment", i+1, "| Size:", len(survey_data))
    print(generate_avatar(my_sk, survey_data))
    print("-------------")

Segment 1 | Size: 12
# Predominant Customer Profile

**Job Title:** Technical Founder/Product Leader

**Desired Outcomes:** Gain practical, hands-on experience building and fine-tuning LLMs for real-world applications, including understanding RAG systems, prompt engineering, and creating AI agents to solve business problems.
-------------
Segment 2 | Size: 4
# Predominant Customer Profile

**Job Title:** Product Leader / Manager

**Desired Outcomes:** Gain hands-on AI/GenAI knowledge to understand technical implementation details, build practical projects, and leverage AI automation capabilities in their product role.
-------------
Segment 3 | Size: 16
# Predominant Customer Profile

**Job Title:** Data Scientist / Technical Professional

**Desired Outcomes:** Gain hands-on AI and Python skills to advance their career, bridge knowledge gaps in modern AI tools and LLMs, and apply AI practically in their current work or build AI-powered applications and businesses.
-------------
Segment 